# Phase 4: Hyperparameter Tuning (Selected Features Track)
**Client:** Crédit Nationale Azur
**Objective:** Use GridSearchCV to optimise the parameters of our Selected Features (k=7) Random Forest and Support Vector Machine models to maximise the F1-Score.

In this notebook, we recreate our strict data preparation pipeline with **SelectKBest feature selection (k=7)** to deploy a systematic grid search across both algorithms using 5-fold cross-validation.

In [ ]:
# Required Imports for Tuning
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

## 1. Load Data and Prepare Targets
We load the clean data, map the target variable to binary integers, and perform our 80/20 stratified split to isolate our training data for the grid search.

In [ ]:
# Load and map target
df = joblib.load('../data/cleaned_data.pkl')
df['personal_loan'] = df['personal_loan'].map({'yes': 1, 'no': 0})
X = df.drop(['personal_loan', 'customer_id'], axis=1)
y = df['personal_loan']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train = X_train.copy()

print(f"Data loaded and split. X_train: {X_train.shape}, X_test: {X_test.shape}")

## 2. One-Hot Encoding
We systematically encode all categorical text variables into integers.

In [ ]:
# Identify and encode categorical columns
categorical_cols = ['education_level', 'credit_card_acct', 'online_acct']

for col in categorical_cols:
    dummies_train = pd.get_dummies(X_train[col], prefix=col, dtype=int)
    X_train = pd.concat([X_train, dummies_train], axis=1).drop([col], axis=1)

# Align test set
X_test, X_train = X_test.align(X_train, join='right', axis=1, fill_value=0)

print("Categorical encoding complete.")

## 3. Feature Selection (SelectKBest with Chi-Square)
We apply SelectKBest(chi2, k=7) to identify the 7 most predictive features before tuning.

In [ ]:
# Apply SelectKBest to identify the k=7 best features
selector = SelectKBest(score_func=chi2, k=7)
X_train_selected_array = selector.fit_transform(X_train, y_train)
X_test_selected_array = selector.transform(X_test)

# Retrieve the names of the selected features
selected_features = X_train.columns[selector.get_support()]
print(f"Selected Features (k=7): {list(selected_features)}")

# Convert the numpy array back to DataFrame
X_train_selected = pd.DataFrame(X_train_selected_array, columns=selected_features, index=X_train.index)
X_test_selected = pd.DataFrame(X_test_selected_array, columns=selected_features, index=X_test.index)

## 4. Standardisation
We apply `StandardScaler` to all continuous features within our selected training set. This is essential for SVM (distance-based) and improves RF as well.

In [ ]:
# Define continuous features from original dataset
continuous_features = ['age', 'credit_amt_used', 'credit_limit', 'family_size', 'income', 'months_customer']

# Filter to only those in selected features
continuous_selected = [col for col in continuous_features if col in X_train_selected.columns]

# Apply StandardScaler to continuous features only
for col in continuous_selected:
    scaler = StandardScaler()
    X_train_selected[col] = scaler.fit_transform(X_train_selected[[col]].values)
    X_test_selected[col] = scaler.transform(X_test_selected[[col]].values)

print(f"Standardisation complete. Ready for tuning.")

## 5. Tuning the Random Forest (Selected Features)
We will test different numbers of trees (`n_estimators`), tree depths (`max_depth`), and split criteria (`criterion`) using 5-fold cross-validation on the training set.

In [ ]:
# Define the parameter grid
rf_param_grid = [
    {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'criterion': ['gini', 'entropy']
    }
]

# Instantiate and fit GridSearchCV
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42), 
    rf_param_grid, 
    cv=5, 
    scoring='f1', 
    verbose=1
)

print("Tuning Random Forest (Selected Features)...")
rf_grid.fit(X_train_selected, y_train)

print(f"\nBest RF Parameters: {rf_grid.best_params_}")
print(f"Best RF F1-Score (CV): {rf_grid.best_score_:.4f}")

## 6. Tuning the Support Vector Machine (Selected Features)
As required, we use `np.arange` to generate float ranges for the regularisation parameter (`C`), alongside testing different kernels and class weights using 5-fold cross-validation.

In [ ]:
# Define the parameter grid using np.arange
svm_param_grid = [
    {
        'C': np.arange(0.5, 2.5, 0.5), 
        'kernel': ['linear', 'rbf'],
        'class_weight': [None, 'balanced']
    }
]

# Instantiate and fit GridSearchCV
svm_grid = GridSearchCV(
    SVC(random_state=42), 
    svm_param_grid, 
    cv=5, 
    scoring='f1', 
    verbose=1
)

print("Tuning Support Vector Machine (Selected Features)...")
svm_grid.fit(X_train_selected, y_train)

print(f"\nBest SVM Parameters: {svm_grid.best_params_}")
print(f"Best SVM F1-Score (CV): {svm_grid.best_score_:.4f}")